In [ ]:
import numpy as np
from os import listdir
import matplotlib.pyplot as plt
import plotly.express as px


def to_difficulty(x, p=1 / 64):
    return np.log(np.select([x > p], [(1 - p) / (1 - x)], x / p))


def from_difficulty(x, p=1 / 64):
    x = np.exp(x)
    return np.select([x > 1], [1 - (1 - p) / x], x * p)


def search(a, v):
    sort = a.argsort(None)
    return sort[np.searchsorted(a.flat[sort], v)]


def max_of_its_ranks(v, *rank):
    ranks = []
    for i, rank in enumerate(rank):
        sort = rank.argsort()
        ranks.append(rank[sort])
        v = np.maximum.accumulate(v[(slice(None),) * i + (sort,)], i)
    return v, ranks


def argmax_of_its_ranks(a, v, *rank):
    vmax = v
    ranks = []
    for i, rank in enumerate(rank):
        sort = rank.argsort()
        ranks.append(rank[sort])
        vmax = np.maximum.accumulate(vmax[(slice(None),) * i + (sort,)], i)
    return a.reshape(-1, *a.shape[v.ndim :])[search(v, vmax)], vmax, ranks


def max_of_its_rank(v, rank):
    return ((rank[:, None] >= rank).all(2) * v).max(1)


def imshow(a, x, y):
    plt.imshow(a, extent=(y[0], y[-1], x[-1], x[0]), aspect="auto")
    plt.colorbar()
    plt.show()


a = np.stack(
    [
        np.concatenate(((int(i[0]), int(0.064 * int(i[1:5])), int(0.012 * int(i[5:9]))), (lambda x: x[::2] if len(x) > 50 else x)(np.fromfile("nnTrainEval2/" + i, "2f")[:, 1])))
        for i in listdir("nnTrainEval2")
    ]
)


In [ ]:
def show_max_of_its_ranks(v, *rank):
    v, rank = max_of_its_ranks(v, *rank)
    plt.imshow(v, extent=(rank[1][0], rank[1][-1], rank[0][-1], rank[0][0]), aspect="auto")
    plt.colorbar()
    plt.show()


show_max_of_its_ranks(a[:, 3:], a[:, :3].prod(1), np.arange(50))
show_max_of_its_ranks(a[:, 3:], a[:, 0], np.arange(50))
show_max_of_its_ranks(a[:, 3:], a[:, 1], np.arange(50))
show_max_of_its_ranks(a[:, 3:], a[:, 2], np.arange(50))

In [ ]:
from sklearn.svm import SVR

x = np.mgrid[1:7, 32:513, 6:109].transpose(1, 2, 3, 0)
y = SVR().fit(a[:, :3], max_of_its_rank(a[:,-1], a[:,:3])).predict(x.reshape(-1, 3)).reshape(x.shape[:3])
for i in y:
    plt.imshow(i, aspect="auto", extent=(32, 512, 108, 6))
    plt.colorbar()
    plt.show()